In [2]:
# EXPERIMENT 7
# Linear Regression, Feature Scaling, and Encoding

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# --------------------------------------------------
# 1. Load Dataset
# --------------------------------------------------

df = pd.read_csv(
    "/content/placement_predict_50k Dataset (1).csv"
)

print("FIRST 5 ROWS")
print(df.head())


# --------------------------------------------------
# 2. Remove Duplicates
# --------------------------------------------------

df = df.drop_duplicates()

print("\nDATASET SHAPE")
print(df.shape)


# --------------------------------------------------
# 3. Select Target
# --------------------------------------------------

if "Salary" in df.columns:

    target_column = "Salary"

else:

    numeric_columns = df.select_dtypes(
        include=["int64", "float64"]
    ).columns

    target_column = numeric_columns[-1]


print("\nTARGET COLUMN")
print(target_column)


# --------------------------------------------------
# 4. Separate X and y
# --------------------------------------------------

X = df.drop(
    columns=[target_column]
)

y = df[target_column]


# --------------------------------------------------
# 5. Convert Target to Numeric
# --------------------------------------------------

y = pd.to_numeric(
    y,
    errors="coerce"
)

valid_rows = y.notna()

X = X.loc[valid_rows]

y = y.loc[valid_rows]


# --------------------------------------------------
# 6. Identify Columns
# --------------------------------------------------

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


print("\nNUMERICAL FEATURES")
print(numerical_columns)

print("\nCATEGORICAL FEATURES")
print(categorical_columns)


# --------------------------------------------------
# 7. Numerical Pipeline
# --------------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)


# --------------------------------------------------
# 8. Categorical Pipeline
# --------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# --------------------------------------------------
# 9. Column Transformer
# --------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)


# --------------------------------------------------
# 10. Linear Regression Model
# --------------------------------------------------

model = LinearRegression()


# --------------------------------------------------
# 11. Complete Pipeline
# --------------------------------------------------

pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),

        (
            "regression",
            model
        )
    ]
)


# --------------------------------------------------
# 12. Train-Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# --------------------------------------------------
# 13. Train Model
# --------------------------------------------------

pipeline.fit(
    X_train,
    y_train
)


# --------------------------------------------------
# 14. Make Predictions
# --------------------------------------------------

y_pred = pipeline.predict(
    X_test
)


# --------------------------------------------------
# 15. Evaluation
# --------------------------------------------------

mse = mean_squared_error(
    y_test,
    y_pred
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)


print("\nMODEL PERFORMANCE")

print("Mean Squared Error :", mse)

print("Mean Absolute Error:", mae)

print("R2 Score           :", r2)


# --------------------------------------------------
# 16. Display Actual vs Predicted
# --------------------------------------------------

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\nACTUAL VS PREDICTED")

print(
    comparison.head(10)
)


print("\nEXPERIMENT 7 COMPLETED")

FIRST 5 ROWS
   StudentID  Gender       City CollegeTier Stream Specialisation Hostel  \
0          1    Male  Ahmedabad       Tier2    ECE     Networking     No   
1          2  Female     Mumbai       Tier2    ECE    DataScience    Yes   
2          3    Male    Kolkata       Tier2     IT    DataScience    Yes   
3          4    Male     Jaipur       Tier1     CS             AI     No   
4          5    Male       Pune       Tier2     IT    DataScience    Yes   

  HistoryOfBacklogs  SGPA_Sem1  SGPA_Sem2  ...  Publications  \
0                No       6.02       6.54  ...             0   
1               Yes       5.84       5.12  ...             0   
2                No       4.91       5.29  ...             0   
3                No       7.67       8.03  ...             0   
4                No       8.14       8.97  ...             1   

   AptitudeTestScore  SoftSkillsRating  CodingTestScore  MockInterviewScore  \
0               66.7               2.2             49.4           